In [15]:
#import
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import torch
from torch.utils.data import TensorDataset, DataLoader
from tarnet import Tarnet
import sys
from pathlib import Path
project_root = Path("/home/ducvu0904/Documents/Lab/RERUM")
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))
from utils import seed_everything
from metrics import auuc, auqc, lift, krcc

In [16]:
%time train_df = pd.read_csv(r"../dataset/Hillstrom/Men/train_men.csv")
%time test_df =  pd.read_csv(r"../dataset/Hillstrom/Men/test_men.csv")
%time val_df = pd.read_csv(r"../dataset/Hillstrom/Men/val_men.csv")

CPU times: user 21.7 ms, sys: 7.01 ms, total: 28.7 ms
Wall time: 28.5 ms
CPU times: user 12.8 ms, sys: 4 ms, total: 16.8 ms
Wall time: 16.7 ms
CPU times: user 4.15 ms, sys: 0 ns, total: 4.15 ms
Wall time: 4.15 ms


In [17]:
in_features = ['recency', 'history_segment', 'history', 'mens', 'womens',
       'zip_code', 'newbie', 'channel_Multichannel', 'channel_Phone', 'channel_Web']
label_feature = ['spend']
treatment_feature = ['treatment']

In [18]:
X_train = train_df[in_features].values.astype(float) # type: ignore
y_train = train_df[label_feature].values.astype(float) # type: ignore
t_train = train_df[treatment_feature].values.astype(float) # type: ignore

X_test = test_df[in_features].values.astype(float) # type: ignore
y_test = test_df[label_feature].values.astype(float) # type: ignore
t_test = test_df[treatment_feature].values.astype(float) # type: ignore

X_val = val_df[in_features].values.astype(float) # type: ignore
y_val = val_df[label_feature].values.astype(float) # type: ignore
t_val = val_df[treatment_feature].values.astype(float) # type: ignore

In [19]:
# Transform to tensor
def to_tensor(arr):
    return torch.tensor(arr, dtype=torch.float32)

x_men_train_t = to_tensor(X_train)
x_men_val_t = to_tensor(X_val)
x_men_test_t = to_tensor(X_test)

y_men_train_t = to_tensor(y_train).reshape(-1, 1)
y_men_val_t = to_tensor(y_val).reshape(-1, 1)
y_men_test_t = to_tensor(y_test).reshape(-1, 1)

# t_train/t_val/t_test cũng tương tự
t_men_train_t = to_tensor(t_train.astype(float)).reshape(-1, 1)
t_men_val_t = to_tensor(t_val.astype(float)).reshape(-1, 1)
t_men_test_t = to_tensor(t_test.astype(float)).reshape(-1, 1)

# Data loader
train_dataset = TensorDataset(x_men_train_t, t_men_train_t, y_men_train_t)
val_dataset = TensorDataset(x_men_val_t, t_men_val_t, y_men_val_t)
test_dataset = TensorDataset(x_men_test_t, t_men_test_t, y_men_test_t)

batch_size = 6400
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=0, pin_memory = True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=0, pin_memory = True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=0, pin_memory = True)

print("-------------------------------------------------------------")
print("✅ Completed transform to tensor ✅")
print(f"Shape of train: x={x_men_train_t.shape}; y={y_men_train_t.shape}; t={t_men_train_t.shape}")
print(f"Shape of val: x={x_men_val_t.shape}; y={y_men_val_t.shape}; t={t_men_val_t.shape}")
print(f"Shape of test: x={x_men_test_t.shape}; y={y_men_test_t.shape}; t={t_men_test_t.shape}")

-------------------------------------------------------------
✅ Completed transform to tensor ✅
Shape of train: x=torch.Size([25567, 10]); y=torch.Size([25567, 1]); t=torch.Size([25567, 1])
Shape of val: x=torch.Size([4262, 10]); y=torch.Size([4262, 1]); t=torch.Size([4262, 1])
Shape of test: x=torch.Size([12784, 10]); y=torch.Size([12784, 1]); t=torch.Size([12784, 1])


In [20]:
epochs = 150
early_stop_metric = "loss"
ema = True
ema_alpha = 0.15
patience = 20
early_stop_start = 30
print (f" epochs = {epochs}")
print (f" early stop = {early_stop_metric}")
print (f" use ema = {ema}")
print (f" ema alpha = {ema_alpha}")
print (f" patience = {patience}")
print (f" early stop start = {early_stop_start}")

 epochs = 150
 early stop = ema_qini
 use ema = True
 ema alpha = 0.15
 patience = 20
 early stop start = 30


In [ ]:
import pandas as pd
import numpy as np
import torch
import io
import optuna
from contextlib import redirect_stdout, redirect_stderr

# Minimize Optuna console noise
optuna.logging.set_verbosity(optuna.logging.WARNING)

# 1. Optuna search config (validation only)
seeds = [412312, 42, 1874, 902745, 1]
n_trials = 50
tpe_sampler_seed = 412312
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def objective(trial):
    grid_lr = trial.suggest_float("lr", 5e-5, 5e-4, log=True)
    grid_wd = trial.suggest_float("weight_decay", 1e-5, 1e-4, log=True)
    grid_outcome_dropout = trial.suggest_float("outcome_dropout", 0.0, 0.5)
    grid_shared_dropout = trial.suggest_float("shared_dropout", 0.0, 0.5)
    grid_outcome_hidden = trial.suggest_int("outcome_hidden", 50, 400, step=50)
    grid_shared_hidden = trial.suggest_int("shared_hidden", 50, 400, step=50)

    val_losses = []
    for SEED in seeds:
        seed_everything(SEED)

        tarnet = Tarnet(
            input_dim=x_men_train_t.shape[1],
            epochs=epochs,
            learning_rate=grid_lr,
            weight_decay=grid_wd,
            use_ema=ema,
            ema_alpha=ema_alpha,
            patience=patience,
            shared_hidden=grid_shared_hidden,
            outcome_hidden=grid_outcome_hidden,
            outcome_dropout=grid_outcome_dropout,
            shared_dropout=grid_shared_dropout,
            early_stop_metric=early_stop_metric,
            early_stop_start_epoch=early_stop_start,
        )

        with redirect_stdout(io.StringIO()), redirect_stderr(io.StringIO()):
            tarnet.fit(train_loader, val_loader)

        current_val_loss = tarnet.validate(val_loader)
        val_losses.append(current_val_loss)

    mean_val_loss = float(np.mean(val_losses))
    std_val_loss = float(np.std(val_losses, ddof=1)) if len(val_losses) > 1 else 0.0
    trial.set_user_attr("std_Val_Loss", std_val_loss)
    return mean_val_loss

def print_trial_callback(study, trial):
    value = float(trial.value) if trial.value is not None else float("nan")
    best_trial = study.best_trial
    best_value = float(best_trial.value) if best_trial.value is not None else float("nan")
    print(
        f"Finished trial {trial.number}: val loss: {value:.4f} - "
        f"with hyperparameters: {trial.params} | "
        f"best trial: {best_trial.number} loss: {best_value:.4f}",
        flush=True
    )

sampler = optuna.samplers.TPESampler(seed=tpe_sampler_seed)
study = optuna.create_study(direction="minimize", sampler=sampler)
study.optimize(objective, n_trials=n_trials, show_progress_bar=True, callbacks=[print_trial_callback])

trial_rows = []
for t in study.trials:
    if t.value is None:
        continue
    trial_rows.append({
        "trial": t.number,
        "lr": round(float(t.params["lr"]), 4),
        "weight_decay": round(float(t.params["weight_decay"]), 4),
        "shared_hidden": int(t.params["shared_hidden"]),
        "outcome_hidden": int(t.params["outcome_hidden"]),
        "shared_dropout": round(float(t.params["shared_dropout"]), 3),
        "outcome_dropout": round(float(t.params["outcome_dropout"]), 3),
        "mean_Val_Loss": float(t.value),
        "std_Val_Loss": float(t.user_attrs.get("std_Val_Loss", np.nan))
    })

df_grid = pd.DataFrame(trial_rows).sort_values("mean_Val_Loss", ascending=True).reset_index(drop=True)
best_params = study.best_params
best_cfg = pd.Series({
    "lr": float(best_params["lr"]),
    "weight_decay": float(best_params["weight_decay"]),
    "shared_hidden": int(best_params["shared_hidden"]),
    "outcome_hidden": int(best_params["outcome_hidden"]),
    "shared_dropout": float(best_params["shared_dropout"]),
    "outcome_dropout": float(best_params["outcome_dropout"]),
    "mean_Val_Loss": float(study.best_value),
    "std_Val_Loss": float(study.best_trial.user_attrs.get("std_Val_Loss", np.nan))
})

  0%|          | 0/50 [00:00<?, ?it/s]

Finished trial 0: val AUQC: 0.8794 - with hyperparameters: {'lr': 0.00015006912017686197, 'weight_decay': 2.3884552609633584e-05, 'outcome_dropout': 0.2041371296334657, 'shared_dropout': 0.2216925330468017, 'outcome_hidden': 250, 'shared_hidden': 350} | best trial: 0 AUQC: 0.8794


Best trial: 0. Best value: 0.879399:   2%|▏         | 1/50 [01:34<1:17:06, 94.41s/it]

Finished trial 1: val AUQC: 0.8728 - with hyperparameters: {'lr': 0.00043025465630583485, 'weight_decay': 9.52546685451366e-05, 'outcome_dropout': 0.29511631137958266, 'shared_dropout': 0.369412397917029, 'outcome_hidden': 350, 'shared_hidden': 150} | best trial: 0 AUQC: 0.8794


Best trial: 0. Best value: 0.879399:   4%|▍         | 2/50 [03:22<1:21:54, 102.39s/it]

...

Best trial: 42. Best value: 0.922902:  96%|█████████▌| 48/50 [1:28:19<03:25, 102.91s/it]

Finished trial 48: val AUQC: 0.9091 - with hyperparameters: {'lr': 0.00019186231227141324, 'weight_decay': 1.0009995423403453e-05, 'outcome_dropout': 0.4713437511360169, 'shared_dropout': 0.029841657890615603, 'outcome_hidden': 50, 'shared_hidden': 400} | best trial: 42 AUQC: 0.9229


Best trial: 42. Best value: 0.922902:  98%|█████████▊| 49/50 [1:30:12<01:45, 105.86s/it]

Finished trial 49: val AUQC: 0.8923 - with hyperparameters: {'lr': 0.0003184539734126092, 'weight_decay': 2.136928997701466e-05, 'outcome_dropout': 0.4811778648384929, 'shared_dropout': 0.3121309461583228, 'outcome_hidden': 50, 'shared_hidden': 200} | best trial: 42 AUQC: 0.9229


Best trial: 42. Best value: 0.922902: 100%|██████████| 50/50 [1:32:30<00:00, 111.00s/it]


In [ ]:
from IPython.display import display

if 'study' not in globals():
    print('Run Cell 10 (Optuna tuning) first.')
else:
    print(f"Best trial: {study.best_trial.number}")
    print(f"Best mean_Val_Loss: {study.best_value:.6f}")
    print(f"Best params: {study.best_params}")

if 'best_cfg' in globals():
    print('\nBest config table:')
    display(best_cfg.to_frame().T)
else:
    print('\nbest_cfg not found.')

if 'df_grid' in globals():
    print('\nTop 10 trials:')
    display(df_grid.head(10))
else:
    print('\ndf_grid not found.')

if 'df_results' in globals():
    print('\nPer-seed test results:')
    display(df_results)
    print('\nTest metrics mean ± std:')
    display(df_results.drop(columns='Seed').agg(['mean', 'std']))

Best trial: 42
Best mean_Val_AUQC: 0.922902
Best params: {'lr': 0.0002738877660032207, 'weight_decay': 1.0002138762693344e-05, 'outcome_dropout': 0.3849899546103965, 'shared_dropout': 0.2632576796060926, 'outcome_hidden': 100, 'shared_hidden': 400}

Best config table:


,lr,weight_decay,shared_hidden,outcome_hidden,shared_dropout,outcome_dropout,mean_Val_AUQC,std_Val_AUQC
0,0.000274,0.00001,400.0,100.0,0.263258,0.38499,0.922902,0.016049



Top 10 trials:


,trial,lr,weight_decay,shared_hidden,outcome_hidden,shared_dropout,outcome_dropout,mean_Val_AUQC,std_Val_AUQC
0,42,0.0003,0.0,400,100,0.263,0.385,0.922902,0.016049
1,47,0.0003,0.0,400,50,0.119,0.458,0.917928,0.021003
2,14,0.0003,0.0,400,100,0.071,0.383,0.917640,0.016629
3,44,0.0003,0.0,400,50,0.269,0.437,0.917585,0.018796
4,28,0.0003,0.0,400,100,0.425,0.346,0.914238,0.024730
5,15,0.0001,0.0,400,100,0.148,0.381,0.910481,0.019284
6,31,0.0004,0.0,400,100,0.264,0.386,0.910371,0.018326
7,48,0.0002,0.0,400,50,0.030,0.471,0.909107,0.014016
8,16,0.0001,0.0,400,100,0.175,0.374,0.909012,0.018227
9,41,0.0001,0.0,400,100,0.164,0.372,0.907971,0.021958


In [23]:
import pandas as pd
import numpy as np
import torch

# 1. Evaluate selected config on test set (after tuning)
seeds = [412312, 42, 1874, 902745, 1]
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
all_runs = []

if 'best_cfg' not in globals():
    raise ValueError("best_cfg not found. Run grid-search cell first.")

best_lr = float(best_cfg['lr'])
best_wd = float(best_cfg['weight_decay'])
best_shared_hidden = int(best_cfg['shared_hidden'])
best_outcome_hidden = int(best_cfg['outcome_hidden'])
best_shared_dropout = float(best_cfg['shared_dropout'])
best_outcome_dropout = float(best_cfg['outcome_dropout'])

print("Evaluating on test with best validation config:")
print(f"  lr={best_lr:.1e}, weight_decay={best_wd:.1e}")
print(f"  shared_hidden={best_shared_hidden}, outcome_hidden={best_outcome_hidden}")
print(f"  shared_dropout={best_shared_dropout:.3f}, outcome_dropout={best_outcome_dropout:.3f}")
print(f"Number of seeds: {len(seeds)}")

# 2. Loop over seeds for robust test evaluation
for SEED in seeds:
    seed_everything(SEED)

    tarnet = Tarnet(
        input_dim=x_men_train_t.shape[1],
        epochs=epochs,
        learning_rate=best_lr,
        weight_decay=best_wd,
        use_ema=ema,
        ema_alpha=ema_alpha,
        patience=patience,
        shared_hidden=best_shared_hidden,
        outcome_hidden=best_outcome_hidden,
        outcome_dropout=best_outcome_dropout,
        shared_dropout=best_shared_dropout,
        early_stop_metric=early_stop_metric,
        early_stop_start_epoch=early_stop_start,
    )

    tarnet.fit(train_loader, val_loader)

    # Test prediction
    x_men_test_t_on_device = x_men_test_t.to(device)
    y0_pred, y1_pred = tarnet.predict(x_men_test_t_on_device)

    uplift_pred = (y1_pred - y0_pred).detach().cpu().numpy().flatten()
    y_true = y_men_test_t.detach().cpu().numpy().flatten()
    t_true = t_men_test_t.detach().cpu().numpy().flatten()

    # ATE error
    ate_pred = uplift_pred.mean()
    ate_true = y_true[t_true == 1].mean() - y_true[t_true == 0].mean()

    all_runs.append({
        'Seed': SEED,
        'AUUC': auuc(y_true, t_true, uplift_pred, bins=100, plot=False),
        'AUQC': auqc(y_true, t_true, uplift_pred, bins=100, plot=False),
        'Lift': lift(y_true, t_true, uplift_pred, h=0.3),
        'KRCC': krcc(y_true, t_true, uplift_pred, bins=100),
        'ATE_Err': abs(ate_pred - ate_true)
    })
    print(f"  Done Seed {SEED}")

# 3. Aggregate final test metrics
df_results = pd.DataFrame(all_runs)

print("\n" + "=" * 85)
print(f"{'PER-SEED DETAILS (TEST SET)':^85}")
print("=" * 85)
print(df_results.to_string(index=False, formatters={
    'AUUC': '{:,.4f}'.format,
    'AUQC': '{:,.4f}'.format,
    'Lift': '{:,.4f}'.format,
    'KRCC': '{:,.4f}'.format,
    'ATE_Err': '{:,.4f}'.format
}))

mean_res = df_results.drop(columns='Seed').mean()
std_res = df_results.drop(columns='Seed').std()

print("=" * 85)
print(f"{'TEST SUMMARY (MEAN ± STD)':^85}")
print("-" * 85)
for metric in ['AUUC', 'AUQC', 'Lift', 'KRCC', 'ATE_Err']:
    print(f"{metric:<10}: {mean_res[metric]:.4f} ± {std_res[metric]:.4f}")
print("=" * 85)

Evaluating on test with best validation config:
  lr=2.7e-04, weight_decay=1.0e-05
  shared_hidden=400, outcome_hidden=100
  shared_dropout=0.263, outcome_dropout=0.385
Number of seeds: 5
🔃🔃🔃Begin training Tarnet🔃🔃🔃
📊 Early Stop Metric: EMA_QINI
📊 Early Stop Start Epoch: 31
📊 Strategy: Best EMA Qini (alpha=0.15)
   Restore to epoch with highest smoothed (EMA) Qini score
   Patience: 20 epochs
Epoch 1/150 | Loss: 2.8426 | Cls: 2.4967 | Reg: 0.3458 | mu_t: 0.0754 | sigma_t: 0.8002 | mu_c: 0.0289 | sigma_c: 0.7369 | Val Loss: 2.9936 | F1_c: 0.0000 | PR_AUC_c: 0.0072 | F1_t: 0.0000 | PR_AUC_t: 0.0111 | Val Qini: 0.6023 | EMA Qini: 0.6023 | Best EMA: 0.6023 ⭐ NEW BEST EMA
Epoch 2/150 | Loss: 2.8457 | Cls: 2.5280 | Reg: 0.3177 | mu_t: 0.0918 | sigma_t: 0.8342 | mu_c: 0.0389 | sigma_c: 0.7483 | Val Loss: 2.9630 | F1_c: 0.0000 | PR_AUC_c: 0.0076 | F1_t: 0.0370 | PR_AUC_t: 0.0241 | Val Qini: 0.6406 | EMA Qini: 0.6080 | Best EMA: 0.6080 ⭐ NEW BEST EMA
Epoch 3/150 | Loss: 3.0204 | Cls: 2.6431 | R